# Stochastic Population Modeling
## Capstone Project Notebook

In ecology, deterministic models like the logistic equation predict a single, smooth trajectory for population growth. In reality, populations are subject to **random fluctuations** from many sources: unpredictable weather, disease outbreaks, food availability, predator encounters, and chance events in birth and death processes. These fluctuations can have profound consequences — particularly for small populations, where a run of bad luck can drive a species to extinction even when the deterministic model predicts stable persistence.

**Stochastic population models** incorporate randomness explicitly, allowing us to ask questions that deterministic models cannot answer:

- What is the **probability of extinction** over a given time horizon?
- How wide is the **range of possible outcomes** for a population starting at a given size?
- How does **environmental variability** (fluctuating climate, resources) interact with **demographic stochasticity** (random birth/death events)?
- What is the **minimum viable population size** needed to ensure long-term survival with high confidence?

### Types of Stochasticity

Ecologists distinguish two fundamental types of randomness:

- **Demographic stochasticity** — arises from the inherent randomness of individual birth and death events. Important for **small populations** where each individual matters. Diminishes as $N$ grows large (law of large numbers).
- **Environmental stochasticity** — arises from random fluctuations in external conditions (weather, food supply, habitat quality) that affect all individuals simultaneously. Important at **all population sizes** because it shifts the entire growth rate up or down.

A third category, **catastrophes** (rare but severe events like floods, fires, or epidemics), can be modelled as occasional large perturbations.

### Mathematical Approach

The simplest way to add stochasticity to the logistic model is via a **stochastic differential equation (SDE)**:

$$dN = rN\left(1 - \frac{N}{K}\right)dt + \sigma N\, dW_t$$

where $dW_t$ is a Wiener process increment (Brownian motion) and $\sigma$ controls the noise intensity. This is solved numerically using the **Euler-Maruyama method**.

### In this notebook you will:
1. Review the deterministic **logistic growth** model
2. Add **stochastic noise** to the growth rate, carrying capacity, or both
3. Understand how different **probability distributions** affect outcomes
4. Visualise ensemble trajectories and **extinction probability**
5. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for numerical computation and random number generation, and Matplotlib for plotting. Stochastic simulations require drawing random numbers at every time step, so NumPy's `default_rng` generator will be used throughout for reproducibility and performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 12})
print('Imports loaded.')

---
## 1 · Deterministic Logistic Growth

$$\frac{dN}{dt} = rN\left(1 - \frac{N}{K}\right)$$

This is our baseline — no randomness, perfectly predictable. Given initial population $N_0$, growth rate $r$, and carrying capacity $K$, the logistic model produces a single sigmoid trajectory that converges monotonically to $K$. Every run with the same parameters gives exactly the same curve.

We implement it with Euler's method and plot the result. This deterministic trajectory will serve as the reference against which all stochastic variants are compared — any deviation from this smooth curve is due to randomness.

In [ ]:
def simulate_logistic(N0, r, K, T, dt=0.1):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    N = np.zeros(steps)
    N[0] = N0
    for k in range(1, steps):
        dN = r * N[k-1] * (1 - N[k-1] / K)
        N[k] = max(0, N[k-1] + dt * dN)
    return t, N

t_det, N_det = simulate_logistic(10, r=0.5, K=100, T=40)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_det, N_det, 'k-', lw=2)
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('Deterministic Logistic Growth', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 2 · Adding Environmental Stochasticity

Real environments are noisy. We model this by adding random perturbations at each time step using a **stochastic differential equation (SDE)**.

### Approach: Euler-Maruyama Method

$$dN = rN\left(1 - \frac{N}{K}\right)dt + \sigma N\, dW_t$$

where $dW_t \sim \mathcal{N}(0, dt)$ is a Wiener increment and $\sigma$ is the noise intensity. The noise is **multiplicative** — it scales with $N$, so larger populations experience larger absolute fluctuations (but the same relative variability).

In discrete form (Euler-Maruyama):

$$N_{t+1} = N_t + r N_t\left(1 - \frac{N_t}{K}\right)\Delta t + \sigma N_t \sqrt{\Delta t}\, \xi_t$$

where $\xi_t \sim \mathcal{N}(0, 1)$. The `max(0, ...)` clamp prevents biologically meaningless negative populations.

We verify the implementation by confirming that with $\sigma = 0$ the stochastic solver exactly reproduces the deterministic logistic curve.

In [ ]:
def simulate_stochastic(N0, r, K, sigma, T, dt=0.1, seed=None):
    """Stochastic logistic growth with multiplicative noise."""
    rng = np.random.default_rng(seed)
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    N = np.zeros(steps)
    N[0] = N0
    for k in range(1, steps):
        drift = r * N[k-1] * (1 - N[k-1] / K) * dt
        diffusion = sigma * N[k-1] * np.sqrt(dt) * rng.standard_normal()
        N[k] = max(0, N[k-1] + drift + diffusion)
    return t, N

# Test: with sigma=0, should match deterministic
t_s0, N_s0 = simulate_stochastic(10, 0.5, 100, sigma=0, T=40, seed=42)
assert np.allclose(N_s0, N_det, atol=0.01), 'Zero noise should match deterministic'
print('Stochastic simulation test passed.')

### Visualise: deterministic vs stochastic

Below we overlay 20 stochastic trajectories (blue) against the deterministic baseline (black dashed) for three noise levels. This is the key diagnostic for understanding how noise affects population dynamics:

- **$\sigma = 0.05$** (low noise) — trajectories cluster tightly around the deterministic curve. The population reliably reaches $K$ and stays near it.
- **$\sigma = 0.15$** (moderate noise) — trajectories spread noticeably. Some runs overshoot $K$, others temporarily dip. The mean still tracks the deterministic curve, but individual runs show clear variability.
- **$\sigma = 0.40$** (high noise) — trajectories diverge dramatically. Some populations explode temporarily; others crash toward zero. Extinction becomes a real possibility even though the deterministic model predicts stable persistence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sigmas = [0.05, 0.15, 0.40]

for ax, sig in zip(axes, sigmas):
    for seed in range(20):
        t_s, N_s = simulate_stochastic(10, 0.5, 100, sigma=sig, T=40, seed=seed)
        ax.plot(t_s, N_s, lw=0.6, alpha=0.4, color='steelblue')
    ax.plot(t_det, N_det, 'k--', lw=2, label='Deterministic')
    ax.set_title(f'$\\sigma = {sig}$', fontweight='bold')
    ax.set_xlabel('Time'); ax.set_ylabel('Population')
    ax.set_ylim(-5, 160); ax.legend(fontsize=9)

plt.suptitle('Effect of Noise Intensity on Population Trajectories', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()
print('Higher noise → wider spread of outcomes → higher extinction risk.')

---
## 3 · Stochastic Carrying Capacity

An alternative approach to environmental noise: instead of perturbing the growth rate directly, let the **carrying capacity** $K$ fluctuate randomly each step:

$$K(t) = K_0 + \epsilon_t, \qquad \epsilon_t \sim \mathcal{N}(0, \sigma_K^2)$$

This models situations where the environment's capacity to support the population changes unpredictably — good years with abundant rainfall alternate with droughts, or habitat quality varies due to land use changes. The population tracks these fluctuations but with a lag, creating persistent oscillations around a noisy equilibrium.

Compared to multiplicative growth-rate noise, stochastic $K$ tends to produce oscillations that are centred around the mean carrying capacity rather than potentially drifting away from it. We clamp $K(t) \geq 1$ to prevent degenerate behaviour.

In [ ]:
def simulate_stochastic_K(N0, r, K0, sigma_K, T, dt=0.1, seed=None):
    """Logistic growth with stochastic carrying capacity."""
    rng = np.random.default_rng(seed)
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    N = np.zeros(steps)
    N[0] = N0
    for k in range(1, steps):
        K_t = max(1, K0 + sigma_K * rng.standard_normal())
        dN = r * N[k-1] * (1 - N[k-1] / K_t) * dt
        N[k] = max(0, N[k-1] + dN)
    return t, N

fig, ax = plt.subplots(figsize=(10, 5))
for seed in range(20):
    t_k, N_k = simulate_stochastic_K(10, 0.5, 100, sigma_K=20, T=80, seed=seed)
    ax.plot(t_k, N_k, lw=0.6, alpha=0.4, color='coral')
ax.plot(t_det, N_det, 'k--', lw=2, label='Deterministic (K=100)')
ax.axhline(100, color='gray', ls=':', lw=1)
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('Stochastic Carrying Capacity ($\\sigma_K = 20$)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()
print('The population fluctuates around K but never settles to a fixed value.')

---
## 4 · Ensemble Statistics

A single stochastic trajectory tells us almost nothing — it is just one possible realisation of a random process. To characterise the system properly, we run a large **ensemble** of simulations (200 runs) with identical parameters but different random seeds, and compute summary statistics across the ensemble at each time point.

The plot below shows the **mean trajectory** (blue line), the **5th–95th percentile envelope** (shaded region capturing 90% of outcomes), and the **deterministic baseline** (black dashed). Key observations:
- The mean of the stochastic ensemble may differ from the deterministic curve due to **Jensen's inequality** — the mean of a nonlinear function of a random variable is not the same as the function of the mean.
- The confidence envelope widens over time as stochastic paths diverge.
- We also count how many runs ended in extinction ($N < 1$ at the final time), giving a direct estimate of extinction probability.

In [ ]:
n_runs = 200
results = []
for seed in range(n_runs):
    _, N_run = simulate_stochastic(10, 0.5, 100, sigma=0.2, T=60, seed=seed)
    results.append(N_run)

results = np.array(results)
mean_N = np.mean(results, axis=0)
q05 = np.percentile(results, 5, axis=0)
q95 = np.percentile(results, 95, axis=0)
t_ens = np.linspace(0, 60, results.shape[1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(t_ens, q05, q95, alpha=0.3, color='steelblue', label='5th–95th percentile')
ax.plot(t_ens, mean_N, 'b-', lw=2, label='Mean')
t_d60, N_d60 = simulate_logistic(10, 0.5, 100, 60)
ax.plot(t_d60, N_d60, 'k--', lw=2, label='Deterministic')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title(f'Ensemble of {n_runs} Stochastic Runs ($\\sigma = 0.2$)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

extinct = np.sum(results[:, -1] < 1)
print(f'Extinction events (N<1 at T=60): {extinct}/{n_runs} = {extinct/n_runs:.1%}')

---
## 5 · Extinction Probability vs Noise

The most consequential question in conservation biology: **how likely is a population to go extinct?** In the deterministic logistic model, any population with $N_0 > 0$ survives forever. Stochastic models tell a very different story.

Below we sweep the noise intensity $\sigma$ from 0 to 0.6 and, for each value, run 200 simulations to estimate the fraction of runs that end in extinction ($N < 1$) by time $T = 100$. The resulting curve shows the **critical noise threshold** — below it, extinction is negligible; above it, extinction probability climbs steeply. This threshold depends on $N_0$, $r$, and $K$: populations that start small, grow slowly, or have low carrying capacity are much more vulnerable to stochastic extinction.

In [ ]:
sigmas_sweep = np.linspace(0, 0.6, 15)
ext_probs = []

for sig in sigmas_sweep:
    extinct_count = 0
    for seed in range(200):
        _, N_run = simulate_stochastic(10, 0.5, 100, sigma=sig, T=100, seed=seed)
        if N_run[-1] < 1:
            extinct_count += 1
    ext_probs.append(extinct_count / 200)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sigmas_sweep, ext_probs, 'ro-', lw=2, markersize=6)
ax.set_xlabel('Noise intensity $\\sigma$')
ax.set_ylabel('Extinction probability')
ax.set_title('Extinction Risk vs Noise Intensity', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 6 · Your Tasks

### Required
1. Implement the logistic equation with stochasticity in $K$, $r$, and both.
2. Use probability distributions (normal, uniform, Poisson) and compare their effects.

### Additional Features (implement at least two)

**Task A: Seasonal Variation**
Make $K$ or $r$ oscillate sinusoidally to model seasons: $K(t) = K_0 + A\sin(2\pi t / T_{\text{year}})$. Add noise on top. Show how seasonality interacts with stochasticity.

**Task B: Catastrophic Events**
At each step, with small probability $p_{\text{cat}}$, reduce $N$ or $K$ drastically (e.g., by 50%). Show recovery dynamics and how catastrophe frequency affects long-term survival.

**Task C: Density-Dependent Variability**
Link noise magnitude to population density: $\sigma(N) = \sigma_0 (N/K)^\alpha$. Analyse how this changes the distribution of outcomes.

### Discussion points
- Compare stochastic vs deterministic trajectories
- Show histograms of final population sizes
- Relate to real ecological examples (insect outbreaks, endangered species)
- Discuss limitations of deterministic models

In [ ]:
# ============================================================
# PLACEHOLDER: Implement your extensions below
# ============================================================
# Task A: Seasonal Variation  — TODO
# Task B: Catastrophic Events — TODO
# Task C: Density-Dependent   — TODO

---
## Recommended Reading & Journal Club

**1. Lande, R. (1993)** *Risks of population extinction from demographic and environmental stochasticity and random catastrophes.* American Naturalist, 142(6), 911–927.
→ Foundational paper distinguishing demographic vs environmental stochasticity.

**2. May, R. M. (1973)** *Stability and Complexity in Model Ecosystems.* Princeton University Press.
→ Classic text on population dynamics, noise, and stability.

**3. Allen, E. (2007)** *Modeling with Itô Stochastic Differential Equations.* Springer.
→ Accessible introduction to SDEs for biologists and ecologists.

**4. Boyce, M. S. et al. (2006)** *Demography in an increasingly variable world.* Trends in Ecology & Evolution, 21(3), 141–148. [DOI](https://doi.org/10.1016/j.tree.2005.11.018)
→ Modern review of environmental stochasticity effects on population viability.

**5. Dennis, B. et al. (1991)** *Estimation of growth and extinction parameters for endangered species.* Ecological Monographs, 61(2), 115–143.
→ Practical application of stochastic population models to conservation.